# 課堂現場：用 LIDA 畫全班的睡眠資料

上台流程（Shift + Enter 逐格執行）：
1. 投影幕（Mini LIDA 網頁 `#wall`）按 **⬇ Download CSV for LIDA**，把 `class_sleep_data.csv` 放進這個資料夾
2. 第 0 格：讀全班資料（找不到檔案時會自動改用 `class_sleep_data_sample.csv` 排練用假資料）
3. Stage 1：LIDA 自己寫資料摘要
4. 🎤 現場提問：把題目（中文英文都可以）填進 `ASK`，LIDA 把它變成分析目標
5. Stage 3：LIDA 自己決定怎麼畫，展開程式碼給大家看
6. Stage 4–5：用一句話改圖、讓 AI 幫自己的圖打分數 → 帶入批判討論

> 臨時想不到題目？把 `ASK` 留空，跑 Stage 2 讓 LIDA 自己出題（備案）。
> 改圖指令（Stage 4）請用**英文**：寫進圖裡的中文會變方框。


In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from lida import Manager, TextGenerationConfig
from gemini_textgen import GeminiTextGenerator

MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
lida = Manager(text_gen=GeminiTextGenerator(model=MODEL))
cfg = TextGenerationConfig(n=1, temperature=0.3, use_cache=True)  # 排練過的輸入會走快取，現場比較快
LIB = "seaborn"   # 作者實驗中 seaborn 失敗率比 matplotlib 低
print("使用模型：", MODEL)

## 0 · 讀全班資料

In [ ]:
DATA = "class_sleep_data.csv"            # 從投影幕下載的檔案
if not os.path.exists(DATA):
    print(f"⚠ 找不到 {DATA}，改用排練用假資料")
    DATA = "class_sleep_data_sample.csv"

df = pd.read_csv(DATA)

# LIDA 的規則摘要會把 "01:30" 這種時間字串排序，得出「最早 00:05、最晚 23:55」這種錯誤結論。
# 預設只留數字欄位；想在課堂上示範這個錯誤，改成 True。
SHOW_CLOCK_TEXT = False
if not SHOW_CLOCK_TEXT:
    df = df.drop(columns=["bedtime", "wakeup_time"])
print(f"{len(df)} 人的資料")
print("bedtime_hour ≥ 24 代表過了午夜（25.5 = 凌晨 1:30）")
df.head()

## Stage 1 · Summarizer：規則算統計，LLM 補上欄位意思

In [ ]:
summary = lida.summarize(df, summary_method="llm", textgen_config=cfg)
print(summary["dataset_description"], "\n")
for f in summary["fields"]:
    p = f["properties"]
    print(f"- {f['column']} ({p['dtype']}): {p.get('description', '')}")

## 🎤 現場提問：你出題，LIDA 把它變成分析目標

把題目寫進 `ASK`，例如：
- `"排出滑手機最長時間+鬧鐘設最多次的人"`
- `"晚睡的人是不是鬧鐘也設比較多？"`

LIDA 的畫圖步驟吃的是「目標」：問題＋建議圖型＋理由（論文的 Goal Explorer 產生的就是這個格式）。這一格請 AI 把你的題目寫成英文目標，畫出來的圖才不會有中文方框。


In [ ]:
ASK = "排出滑手機最長時間+鬧鐘設最多次的人"   # 留空 "" 就改跑 Stage 2 備案

import json, re
from lida.datamodel import Goal

def ask_to_goal(ask):
    columns = "\n".join(f"- {f['column']}: {f['properties'].get('description', '')}" for f in summary["fields"])
    prompt = f"""A presenter asked this question about a dataset, possibly in Chinese:
{ask}

Dataset columns:
{columns}

Rewrite it as ONE visualization goal in English, keeping the presenter's intent
(e.g. "排出" means rank/sort). Use only the columns above.
Reply with JSON only: {{"question": "...", "visualization": "...", "rationale": "..."}}
- question: the English question, short enough to be a chart title
- visualization: the chart type and which column goes on which axis
- rationale: why this chart answers the question"""
    reply = lida.text_gen.generate([{"role": "user", "content": prompt}], config=cfg).text[0]["content"]
    return Goal(**json.loads(re.search(r"\{.*\}", reply, re.S).group()))

if ASK:
    my_goal = ask_to_goal(ASK)
    print(f"你的題目：{ASK}\n")
    print(f"LIDA 的目標：{my_goal.question}\n    圖：{my_goal.visualization}\n    理由：{my_goal.rationale}")


## Stage 2（備案）· Goal Explorer：LIDA 自己提問題

`ASK` 有填題目就可以跳過這格。沒題目時，唸出這幾題，請同學投票（舉手或喊號碼）。


In [ ]:
PERSONA = "a design graduate student who is curious (and a little nosy) about classmates' sleep and phone habits"

goals = lida.goals(summary, n=4, persona=PERSONA, textgen_config=cfg)
for i, g in enumerate(goals):
    print(f"[{i}] {g.question}\n    圖：{g.visualization}\n    理由：{g.rationale}\n")

## Stage 3 · VisGenerator：AI 寫程式，Python 畫圖

有填 `ASK` 就畫你的題目；沒填就畫 Stage 2 第 `PICK` 題。


In [ ]:
PICK = 0   # 只有 ASK 留空時才用到

goal = my_goal if ASK else goals[PICK]
charts = lida.visualize(summary=summary, goal=goal, textgen_config=cfg,
                        library=LIB, return_error=True)
if charts and charts[0].status:
    display(charts[0])
else:
    print("✗ 這次 AI 寫的程式跑不起來（這就是論文裡的 VER 失敗）。再執行一次或換題。")
    if charts: print(charts[0].error)

In [ ]:
# 展開給大家看：這段程式是 AI 剛剛寫的
print(charts[0].code)

## Stage 4 · VizOps：用一句話改圖

可以請同學出修改指令（英文），例如：
- `"make all text larger so older adults can read it"`
- `"highlight the person with the most alarms in orange"`
- `"add a title that tells the main finding"`

In [ ]:
INSTRUCTIONS = ["make all text larger", "highlight the person with the most alarms in orange"]

edited = lida.edit(code=charts[0].code, summary=summary, instructions=INSTRUCTIONS,
                   library=LIB, textgen_config=cfg, return_error=True)
display(edited[0])

## Stage 5 · 解釋與自我評分（SEVQ）

批判點：**AI 畫的圖，由 AI 自己打分數**；而且它只讀程式碼，沒有看到畫出來的圖。

In [ ]:
from lida.datamodel import Goal
g = goal if isinstance(goal, Goal) else Goal(question=goal, visualization=goal, rationale="")

scores = lida.evaluate(code=edited[0].code, goal=g, library=LIB, textgen_config=cfg)[0]
for e in scores:
    print(f"{e['dimension']:<16} {e['score']}/10  {e['rationale'][:110]}")
print("\n平均（SEVQ）：", round(sum(e["score"] for e in scores) / len(scores), 1))

## 討論（2 分鐘）

1. 同一份資料，LIDA 選了這種圖。你會怎麼畫？**畫法該由設計者決定，還是交給 AI？**
2. 只有 18 個人的資料，AI 的結論撐得起來嗎？
3. AI 給自己的圖打了高分，你同意嗎？誰來評分才公平？